# Solution E2 · The same columns, in another order: the model does not notice

**Type**: Find the bug

## Objective

See what happens when a model trained on numpy arrays receives in production a DataFrame with the same columns in a different order: **no error** and almost random predictions. Build a pipeline that selects the columns by name and
a schema checking function.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

wine = load_wine(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(wine.data, wine.target, test_size=0.3, stratify=wine.target, random_state=0)
columns = list(X_tr.columns)
prod_order = list(np.random.default_rng(1).permutation(columns))
X_prod = X_te[prod_order]                                    # same columns and same examples, different order

## Commented solution

In [ ]:
bug_model = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr.to_numpy(), y_tr)
test_acc = bug_model.score(X_te.to_numpy(), y_te)
bug_prod_acc = bug_model.score(X_prod.to_numpy(), y_te)
print(f"test accuracy: {test_acc:.3f} | in production (columns in a different order): {bug_prod_acc:.3f}")

**Where the bug is.** With `.to_numpy()` the model lost the names: at prediction time it reads the column in position 0 as if it were `alcohol`, the one in position 1 as `malic_acid`, and so on. If the columns change order, every feature ends up in the
wrong position and the scaler and the KNN work on numbers of another magnitude. No check intervenes because the shapes coincide.

In [ ]:
changed_positions = sum(a != b for a, b in zip(columns, prod_order))
print("columns in a different position:", changed_positions, "out of", len(columns))

In [ ]:
def check_schema(X, expected):
    missing, extra = set(expected) - set(X.columns), set(X.columns) - set(expected)
    if missing or extra:
        raise ValueError(f"different schema: missing {sorted(missing)}, extra {sorted(extra)}")
    return X[list(expected)]                                           # same order as in training

In [ ]:
ok_model = make_pipeline(ColumnTransformer([("num", StandardScaler(), columns)]), KNeighborsClassifier(7)).fit(X_tr, y_tr)
ok_test_acc = ok_model.score(X_te, y_te)
ok_prod_acc = ok_model.score(X_prod, y_te)
print(f"with selection by name: test {ok_test_acc:.3f} | production {ok_prod_acc:.3f}")

## Check

In [ ]:
with verify():
    check(test_acc > 0.9 and bug_prod_acc < 0.7, "the colleague's model: over 0.9 on the test set, below 0.7 with the shuffled columns, with no errors", "don't modify the colleague's code")
    check(changed_positions == sum(a != b for a, b in zip(columns, prod_order)) and changed_positions >= 10, "at least 10 columns out of 13 are in a different position", "compare the two lists position by position")
    check(list(check_schema(X_prod, columns).columns) == columns and (check_schema(X_prod, columns).to_numpy() == X_te.to_numpy()).all(), "check_schema reorders the columns as in training", "X[list(expected)]")
    for bad, name in [(X_prod.drop(columns=columns[0]), "missing column"), (X_prod.assign(extra=0.0), "extra column")]:
        try:
            check_schema(bad, columns)
            raised = False
        except ValueError:
            raised = True
        check(raised, f"check_schema raises ValueError with a {name}", "compare set(X.columns) with set(expected)")
    check(ok_model is not None and ok_prod_acc > 0.9 and abs(ok_prod_acc - ok_test_acc) < 1e-12, "with the selection by name the shuffled columns give the same result as the test set", "ColumnTransformer([('num', StandardScaler(), columns)])")
done()

## What you learned

- **A model trained on numpy has no names**: it reads the columns by position. If in production the order changes (another query, another system, a column added in the middle) the predictions get ruined with no error at all: here from over 0.9 to about 0.4.
- Structural remedy: **train on DataFrames and select the columns by name** in the `ColumnTransformer`. The order of the incoming data no longer matters (recent scikit-learn versions also check `feature_names_in_` and will raise an error, but your
  pipeline must be correct even when it doesn't: with 1.1 the check is only a warning).
- Second level of defense: an explicit **schema check** at the entrance (expected columns, types, plausible ranges) that fails loudly. In production a clear error is worth far more than a wrong prediction.
- KNN is particularly vulnerable because the distances mix all the columns, but the same holds for a linear regression (coefficients attached to the wrong columns).

## Common pitfall

Converting the DataFrames to arrays (`.values`, `.to_numpy()`) "to be sure scikit-learn works" at the start of the pipeline. It is exactly the step that erases the protection of the names.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you avoid schema errors when a model goes into production?</summary>

I train on DataFrames with selection of the columns by name inside the pipeline, save the list of expected columns with the model and verify the incoming schema (names, types, ranges). I add automated tests with example data in a different order and with missing
columns, and monitor the distribution of the features in production against that of the training set.

</details>